# April Forecast (with headwinds) vs Actuals — Regional Breakdown

Interactive Plotly comparison of actuals (BigQuery, current through yesterday) against the April 2026 desktop forecast for:

1. **World** total
2. **Regions** (named-country subset): North America, South America, Western Europe, Eastern Europe (incl. RU), Asia (excl. RU), Rest of World. Africa and Oceania are skipped — the forecast parquet has no individual-country coverage for them.
3. **Western Europe** — individual country comparisons for DE, FR, IT, plus an actuals-only pane for all WE countries combined.
4. **China**

Each entity gets two panes: 28-day moving average (left) and raw daily DAU (right). All panes use the aggregated ALL-OS desktop slice (segment `{"os": "ALL"}`).

**Forecast source:** `data-official/2026-04/desktop_cps0.15983_thresh050_recent13_clip0.6/mozaic_daily_forecast.2026-04-01.ld-D.raw.parquet`

**Headwinds:** the World pane applies the linear-ramp headwind from `data-official/2026-04/adjustments/headwind.json` (-1,497,870 desktop DAU at 2026-12-15). Regional and country panes show the raw forecast — the headwind is defined at the world-total level only and we don't have a per-region attribution.

**Coverage:** the forecast parquet has 14 named countries (AR, BR, CA, CN, DE, FR, ID, IN, IT, JP, MX, PL, RU, US) plus a single `ROW` aggregate. Regional sums use only the named countries that fall in each region; the actuals are summed across the same country lists so the two series are directly comparable.

In [1]:
# [setup]
import json
import sys
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from google.cloud import bigquery
from plotly.subplots import make_subplots

# --- File paths ---
DESKTOP_FORECAST_PATH = (
    "data-official/2026-04/desktop_cps0.15983_thresh050_recent13_clip0.6/"
    "mozaic_daily_forecast.2026-04-01.ld-D.raw.parquet"
)
HEADWIND_PATH = "data-official/2026-04/adjustments/headwind.json"

# --- Date constants ---
DISPLAY_START = pd.Timestamp("2026-01-01")
DISPLAY_END = pd.Timestamp("2026-12-31")
FORECAST_START = pd.Timestamp("2026-04-01")
BQ_START = "2025-12-04"  # 28 days before DISPLAY_START so the MA has warmup

# Countries that appear individually in the forecast parquet (everything else is in ROW)
FORECAST_COUNTRIES = [
    "AR", "BR", "CA", "CN", "DE", "FR", "ID",
    "IN", "IT", "JP", "MX", "PL", "RU", "US",
]

# Region -> country list (named-country subset only).
# Africa / Oceania have no named-country coverage -> empty list -> skipped.
# Rest of World is special: forecast uses the ROW bucket; actuals use anything NOT in FORECAST_COUNTRIES.
REGIONS = {
    "North America":             ["US", "CA", "MX"],
    "South America":             ["AR", "BR"],
    "Western Europe":            ["DE", "FR", "IT"],
    "Eastern Europe (incl. RU)": ["PL", "RU"],
    "Asia (excl. RU)":           ["CN", "ID", "IN", "JP"],
    "Africa":                    [],
    "Oceania":                   [],
    "Rest of World":             "ROW",
}

# All Western European ISO codes for the actuals-only "all WE" pane
WESTERN_EUROPE_FULL = [
    "AT", "BE", "CH", "DE", "DK", "ES", "FI", "FR", "GB", "GR",
    "IE", "IS", "IT", "LU", "MT", "NL", "NO", "PT", "SE",
]

/Users/brendanwells/work/mozaic-daily/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/Users/brendanwells/work/mozaic-daily/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.bigquery_storage_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.bigquery_storage_v1 past that date.
  warnings.warn(message, FutureWarning)


In [2]:
# [helpers]
def render_headwind_daily(spec, date_index):
    """Render the linear-ramp headwind spec as a daily desktop DAU adjustment series."""
    if spec["type"] != "linear_ramp":
        raise ValueError(f"unsupported headwind spec type: {spec['type']}")
    idx = pd.DatetimeIndex(date_index)
    start = pd.Timestamp(spec["start_date"])
    anchor = pd.Timestamp(spec["anchor_date"])
    total_days = (anchor - start).days
    elapsed = np.maximum(0, (idx - start).days)
    out = pd.Series(spec.get("desktop_dau", 0) * elapsed / total_days, index=idx)
    out[idx < start] = 0.0
    return out


def to_28ma(s):
    """28-day rolling mean on a date-indexed Series."""
    return s.sort_index().rolling(28).mean()


def load_desktop_all_os(path):
    """Load desktop forecast rows for segment={\"os\": \"ALL\"}: per-country + ALL + ROW."""
    df = pd.read_parquet(path)
    mask = df["segment"] == '{"os": "ALL"}'
    out = df.loc[mask, ["target_date", "country", "dau", "data_type"]].copy()
    out["target_date"] = pd.to_datetime(out["target_date"])
    return out.sort_values(["country", "target_date"]).reset_index(drop=True)

In [3]:
# [load-forecast]
forecast_df = load_desktop_all_os(DESKTOP_FORECAST_PATH)
print(f"Loaded {len(forecast_df):,} forecast rows")
print(f"Date range: {forecast_df['target_date'].min().date()} to {forecast_df['target_date'].max().date()}")
print(f"Countries:  {sorted(forecast_df['country'].unique())}")

with open(HEADWIND_PATH) as f:
    headwind_spec = json.load(f)
print(f"\nHeadwind spec: {headwind_spec}")

all_dates = pd.DatetimeIndex(sorted(forecast_df["target_date"].unique()))
headwind_daily = render_headwind_daily(headwind_spec, all_dates)
print(f"\nHeadwind on {FORECAST_START.date()}:        {headwind_daily.loc[FORECAST_START]:+,.0f} DAU")
print(f"Headwind on 2026-12-15 (anchor): {headwind_daily.loc[pd.Timestamp('2026-12-15')]:+,.0f} DAU")

Loaded 46,752 forecast rows
Date range: 2020-01-01 to 2027-12-31
Countries:  ['ALL', 'AR', 'BR', 'CA', 'CN', 'DE', 'FR', 'ID', 'IN', 'IT', 'JP', 'MX', 'PL', 'ROW', 'RU', 'US']

Headwind spec: {'type': 'linear_ramp', 'start_date': '2026-04-01', 'anchor_date': '2026-12-15', 'desktop_dau': -1497870, 'mobile_dau': -27162}

Headwind on 2026-04-01:        +0 DAU
Headwind on 2026-12-15 (anchor): -1,497,870 DAU


In [5]:
# [bq-actuals]
client = bigquery.Client(project="moz-fx-data-bq-data-science")

sql = f"""
SELECT submission_date AS date, country, SUM(dau) AS dau
FROM `moz-fx-data-shared-prod.telemetry.active_users_aggregates`
WHERE app_name = "Firefox Desktop"
  AND submission_date BETWEEN '{BQ_START}' AND CURRENT_DATE() - 1
GROUP BY submission_date, country
ORDER BY country, submission_date
"""

t0 = time.time()
sys.stdout.write("Querying per-country desktop actuals ...")
sys.stdout.flush()
actuals_long = client.query(sql).to_dataframe()
actuals_long["date"] = pd.to_datetime(actuals_long["date"])
sys.stdout.write(
    f" {len(actuals_long):,} rows, {actuals_long['country'].nunique()} countries"
    f" ({time.time() - t0:.1f}s)\n"
)

print(f"Actuals date range: {actuals_long['date'].min().date()} to {actuals_long['date'].max().date()}")

Querying per-country desktop actuals ... 40,696 rows, 247 countries (9.1s)
Actuals date range: 2025-12-04 to 2026-05-20


In [6]:
# [build-entities]
def _sum_forecast(countries):
    sub = forecast_df[forecast_df["country"].isin(countries)]
    return sub.groupby("target_date")["dau"].sum().sort_index()


def _sum_actuals(countries):
    sub = actuals_long[actuals_long["country"].isin(countries)]
    return sub.groupby("date")["dau"].sum().sort_index()


def build_entity(name, *, forecast_source, actuals_source, apply_headwind=False, has_forecast=True):
    """Assemble actual/forecast daily + 28d MA for one plot entity.

    forecast_source: 'ALL' | 'ROW' | list of country codes | None (when has_forecast=False).
    actuals_source:  'ALL' | 'ROW' | list of country codes.
    """
    if has_forecast:
        if forecast_source == "ALL":
            forecast_daily = (
                forecast_df[forecast_df["country"] == "ALL"]
                .set_index("target_date")["dau"].sort_index()
            )
        elif forecast_source == "ROW":
            forecast_daily = (
                forecast_df[forecast_df["country"] == "ROW"]
                .set_index("target_date")["dau"].sort_index()
            )
        else:
            forecast_daily = _sum_forecast(forecast_source)
        if apply_headwind:
            forecast_daily = forecast_daily.add(headwind_daily, fill_value=0)
        forecast_ma = to_28ma(forecast_daily)
    else:
        forecast_daily = None
        forecast_ma = None

    if actuals_source == "ALL":
        actuals_daily = actuals_long.groupby("date")["dau"].sum().sort_index()
    elif actuals_source == "ROW":
        actuals_daily = _sum_actuals(
            [c for c in actuals_long["country"].unique() if c not in FORECAST_COUNTRIES]
        )
    else:
        actuals_daily = _sum_actuals(actuals_source)
    actuals_ma = to_28ma(actuals_daily)

    return {
        "name": name,
        "actual_daily": actuals_daily,
        "actual_ma": actuals_ma,
        "forecast_daily": forecast_daily,
        "forecast_ma": forecast_ma,
        "headwind_applied": apply_headwind,
        "has_forecast": has_forecast,
    }


# World — headwind applied (spec is defined at world-total only)
world = build_entity("World", forecast_source="ALL", actuals_source="ALL", apply_headwind=True)

# Regions
region_entities = []
for region_name, countries in REGIONS.items():
    if countries == "ROW":
        region_entities.append(
            build_entity(region_name, forecast_source="ROW", actuals_source="ROW")
        )
    elif not countries:
        print(f"Skipping {region_name}: no named-country forecast coverage")
        continue
    else:
        region_entities.append(
            build_entity(region_name, forecast_source=countries, actuals_source=countries)
        )

# Western Europe individual countries (the three we have forecasts for)
we_entities = [
    build_entity("Germany (DE)", forecast_source=["DE"], actuals_source=["DE"]),
    build_entity("France (FR)",  forecast_source=["FR"], actuals_source=["FR"]),
    build_entity("Italy (IT)",   forecast_source=["IT"], actuals_source=["IT"]),
]

# Hybrid: all WE countries, actuals only (the forecast parquet only covers DE/FR/IT individually)
we_full = build_entity(
    "Western Europe — all countries (actuals only)",
    forecast_source=None,
    actuals_source=WESTERN_EUROPE_FULL,
    has_forecast=False,
)

# China
china = build_entity("China (CN)", forecast_source=["CN"], actuals_source=["CN"])

print("\nEntity coverage (post-forecast-start mean DAU):")
for e in [world] + region_entities + we_entities + [we_full, china]:
    a = e["actual_daily"]
    f = e["forecast_daily"]
    a_mean = a.loc[a.index >= FORECAST_START].mean() if len(a) else 0
    f_mean = f.loc[f.index >= FORECAST_START].mean() if f is not None else None
    f_str = f"forecast {f_mean:>14,.0f}" if f_mean is not None else "no forecast"
    print(f"  {e['name']:<48} actual {a_mean:>14,.0f}  {f_str}")

Skipping Africa: no named-country forecast coverage
Skipping Oceania: no named-country forecast coverage

Entity coverage (post-forecast-start mean DAU):
  World                                            actual     48,189,336  forecast     43,337,533
  North America                                    actual      9,266,647  forecast      8,679,270
  South America                                    actual      2,347,231  forecast      2,263,824
  Western Europe                                   actual     10,305,683  forecast      9,896,447
  Eastern Europe (incl. RU)                        actual      3,520,148  forecast      3,262,176
  Asia (excl. RU)                                  actual      6,347,593  forecast      5,510,237
  Rest of World                                    actual     16,402,034  forecast     15,580,499
  Germany (DE)                                     actual      5,435,250  forecast      5,185,495
  France (FR)                                      actual     

In [7]:
# [plot-helpers]
COLOR_ACTUAL = "#1f1f1f"
COLOR_FORECAST = "#1f77b4"


def _restrict(series, start=DISPLAY_START, end=DISPLAY_END):
    if series is None:
        return None
    return series.loc[(series.index >= start) & (series.index <= end)]


def _add_pair(fig, row, col, x_a, y_a, x_f, y_f, forecast_label, showlegend):
    fig.add_trace(
        go.Scatter(
            x=x_a, y=y_a, mode="lines", name="Actual",
            line=dict(color=COLOR_ACTUAL, width=2),
            legendgroup="actual", showlegend=showlegend,
            hovertemplate="%{x|%Y-%m-%d}<br>Actual: %{y:,.0f}<extra></extra>",
        ),
        row=row, col=col,
    )
    if x_f is not None:
        fig.add_trace(
            go.Scatter(
                x=x_f, y=y_f, mode="lines", name=forecast_label,
                line=dict(color=COLOR_FORECAST, width=2, dash="dash"),
                legendgroup="forecast", showlegend=showlegend,
                hovertemplate="%{x|%Y-%m-%d}<br>Forecast: %{y:,.0f}<extra></extra>",
            ),
            row=row, col=col,
        )


def plot_entities(entities, title, height_per_row=380):
    """Stacked-subplot figure: one row per entity, MA left, daily right."""
    n = len(entities)
    subtitles = []
    for e in entities:
        suffix = " (+ headwind)" if e["headwind_applied"] else ""
        subtitles.append(f"{e['name']} — 28-day MA{suffix}")
        subtitles.append(f"{e['name']} — Daily{suffix}")

    vspace = min(0.08, 0.7 / max(n, 1)) if n > 1 else 0.08
    fig = make_subplots(
        rows=n, cols=2,
        subplot_titles=subtitles,
        horizontal_spacing=0.07,
        vertical_spacing=vspace,
    )

    for i, e in enumerate(entities):
        row = i + 1
        forecast_label = "Forecast (+ headwind)" if e["headwind_applied"] else "Forecast"
        showlegend = (i == 0)

        a_ma = _restrict(e["actual_ma"])
        f_ma = _restrict(e["forecast_ma"])
        a_d = _restrict(e["actual_daily"])
        f_d = _restrict(e["forecast_daily"])

        _add_pair(
            fig, row=row, col=1,
            x_a=a_ma.index, y_a=a_ma.values,
            x_f=(f_ma.index if f_ma is not None else None),
            y_f=(f_ma.values if f_ma is not None else None),
            forecast_label=forecast_label, showlegend=showlegend,
        )
        _add_pair(
            fig, row=row, col=2,
            x_a=a_d.index, y_a=a_d.values,
            x_f=(f_d.index if f_d is not None else None),
            y_f=(f_d.values if f_d is not None else None),
            forecast_label=forecast_label, showlegend=False,
        )

        fig.add_vline(
            x=FORECAST_START.to_pydatetime(),
            line=dict(color="#888", width=1, dash="dot"),
            row=row, col=1,
        )
        fig.add_vline(
            x=FORECAST_START.to_pydatetime(),
            line=dict(color="#888", width=1, dash="dot"),
            row=row, col=2,
        )

    fig.update_layout(
        height=height_per_row * n + 80,
        width=1300,
        title=title,
        hovermode="x unified",
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(t=100, b=40, l=60, r=40),
    )
    fig.update_xaxes(range=[DISPLAY_START, DISPLAY_END])
    fig.update_yaxes(tickformat="~s")
    return fig

## 1. World total

Sum across all countries. The forecast curve has the linear-ramp headwind applied (reaching -1.5M desktop DAU at 2026-12-15). Vertical dotted line marks the forecast start (2026-04-01).

In [8]:
# [plot-world]
fig = plot_entities([world], "World — Actual vs April Forecast (+ headwind)")
fig.show()

## 2. Regions

Each region's forecast is the sum of the named countries that fall in it; actuals use the same country list. Africa and Oceania are skipped — no named-country coverage. "Rest of World" uses the forecast's `ROW` bucket against actuals from every country *not* in the named subset.

In [9]:
# [plot-regions]
fig = plot_entities(region_entities, "Regions — Actual vs April Forecast (named-country subset)")
fig.show()

## 3a. Western Europe — individual countries

The April forecast only covers Germany, France, and Italy individually — every other Western European country sits inside the `ROW` aggregate.

In [10]:
# [plot-we-countries]
fig = plot_entities(we_entities, "Western Europe — Individual Countries (Actual vs Forecast)")
fig.show()

## 3b. Western Europe — all countries (actuals only)

Sum across all Western European ISO codes (AT, BE, CH, DE, DK, ES, FI, FR, GB, GR, IE, IS, IT, LU, MT, NL, NO, PT, SE). No forecast curve — the parquet doesn't break out the unnamed countries.

In [11]:
# [plot-we-all]
fig = plot_entities([we_full], "Western Europe — All Countries Combined (Actuals Only)")
fig.show()

## 4. China

In [12]:
# [plot-china]
fig = plot_entities([china], "China — Actual vs April Forecast")
fig.show()